# HW2 — Data Engineering at NimbusMegaMart

**Goal:** Build a daily KPI table by `country × category` with total events, purchases, revenue, unique users, and a 7-day rolling revenue. Store outputs as date-partitioned Parquet.

```bash
docker run --rm -it -p 8888:8888 -p 4040:4040 \
-v /Users/shamilarslanov/Projects/Harbour.Space/IntroToDE/HW2:/home/jovyan/work \
-e JUPYTER_TOKEN=letmein \
-d docker.io/jupyter/pyspark-notebook:latest
```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, ArrayType
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("HDFS Demo") \
    .master("local[*]") \
    .config("spark.hadoop.sun.security.auth.login.config", "true") \
    .config("spark.driver.extraJavaOptions", "--add-opens java.base/javax.security.auth=ALL-UNNAMED") \
    .getOrCreate()

spark

## Task 1 — Define schemas using StructType / StructField

In [2]:
# --- Users schema ---
# {"id": 1, "signup_date": "2025-03-16", "plan": "free", "country": "TH", "marketing_opt_in": false}
users_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("signup_date", StringType(), True),
    StructField("plan", StringType(), True),
    StructField("country", StringType(), True),
    StructField("marketing_opt_in", BooleanType(), True),
])

# --- Items schema ---
# {"item_id": 1, "category": "sports", "tags": ["new", "clearance"]}
items_schema = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("category", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
])

# --- Events schema ---
# Nested structs for context, props, exp
context_schema = StructType([
    StructField("country", StringType(), True),
    StructField("device", StringType(), True),
    StructField("locale", StringType(), True),
    StructField("session_id", StringType(), True),
])

props_schema = StructType([
    StructField("price", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("dwell_ms", IntegerType(), True),
])

exp_schema = StructType([
    StructField("ab_group", StringType(), True),
])

events_schema = StructType([
    StructField("ts", StringType(), False),
    StructField("event", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("item_id", IntegerType(), True),
    StructField("context", context_schema, True),
    StructField("props", props_schema, True),
    StructField("exp", exp_schema, True),
])

## Task 2 — Read datasets into DataFrames with explicit schemas

In [3]:
DATA_DIR = "../data"

users_df = spark.read.json(f"{DATA_DIR}/users.jsonl", schema=users_schema)
items_df = spark.read.json(f"{DATA_DIR}/items.jsonl", schema=items_schema)
events_df = spark.read.json(f"{DATA_DIR}/events/", schema=events_schema)

users_df.printSchema()
items_df.printSchema()
events_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- plan: string (nullable = true)
 |-- country: string (nullable = true)
 |-- marketing_opt_in: boolean (nullable = true)

root
 |-- item_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)

root
 |-- ts: string (nullable = true)
 |-- event: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- device: string (nullable = true)
 |    |-- locale: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- props: struct (nullable = true)
 |    |-- price: double (nullable = true)
 |    |-- payment_method: string (nullable = true)
 |    |-- dwell_ms: integer (nullable = true)
 |-- exp: struct (nullable = true)
 |    |-- ab_group: string (nullable = t

## Task 3 — Record counts and partition counts

In [4]:
for name, df in [("users", users_df), ("items", items_df), ("events", events_df)]:
    print(f"{name}: {df.count()} records, {df.rdd.getNumPartitions()} partitions")

users: 20000 records, 1 partitions
items: 5000 records, 1 partitions
events: 200000 records, 4 partitions


## Task 4 — Add `timestamp` and `date` columns from `ts`

In [5]:
events_df = events_df \
    .withColumn("timestamp", F.to_timestamp("ts")) \
    .withColumn("date", F.to_date("ts"))

events_df.select("ts", "timestamp", "date").show(5, truncate=False)

+--------------------------------+--------------------------+----------+
|ts                              |timestamp                 |date      |
+--------------------------------+--------------------------+----------+
|2026-02-16T10:06:22.001323+00:00|2026-02-16 10:06:22.001323|2026-02-16|
|2026-02-09T11:26:23.001323+00:00|2026-02-09 11:26:23.001323|2026-02-09|
|2026-02-26T20:06:44.001323+00:00|2026-02-26 20:06:44.001323|2026-02-26|
|2026-03-10T23:47:16.001323+00:00|2026-03-10 23:47:16.001323|2026-03-10|
|2026-03-17T22:27:50.001323+00:00|2026-03-17 22:27:50.001323|2026-03-17|
+--------------------------------+--------------------------+----------+
only showing top 5 rows



## Task 5 — Create `revenue` column (price for purchases, 0.0 otherwise)

In [6]:
events_df = events_df.withColumn(
    "revenue",
    F.when(F.col("event") == "purchase", F.col("props.price"))
     .otherwise(0.0)
     .cast(DoubleType())
)

events_df.select("event", "props.price", "revenue").show(10)

+-----+-----+-------+
|event|price|revenue|
+-----+-----+-------+
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
| view| NULL|    0.0|
+-----+-----+-------+
only showing top 10 rows



## Task 6 — Filter out rows with negative revenue

In [7]:
before_count = events_df.count()
events_df = events_df.filter(F.col("revenue") >= 0.0)
after_count = events_df.count()

print(f"Removed {before_count - after_count} rows with negative revenue")

Removed 5 rows with negative revenue


## Task 7 — Broadcast join with items and users

`broadcast()` tells Spark to send the entire smaller DataFrame to every executor, avoiding an expensive shuffle of the large events DataFrame. Since users (20K rows) and items (5K rows) are much smaller than events (200K rows), broadcasting them is efficient — each executor can perform a local hash join without any network shuffle of the events data.

In [8]:
# broadcast() sends the small table to all executors so the large events table
# doesn't need to be shuffled across the network — this avoids expensive shuffle stages.
enriched_df = events_df \
    .join(F.broadcast(items_df), on="item_id", how="left") \
    .join(F.broadcast(users_df), events_df["user_id"] == users_df["id"], how="left") \
    .drop(users_df["country"]) \
    .drop(users_df["id"])

enriched_df.printSchema()
enriched_df.show(5)

root
 |-- item_id: integer (nullable = true)
 |-- ts: string (nullable = true)
 |-- event: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- device: string (nullable = true)
 |    |-- locale: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- props: struct (nullable = true)
 |    |-- price: double (nullable = true)
 |    |-- payment_method: string (nullable = true)
 |    |-- dwell_ms: integer (nullable = true)
 |-- exp: struct (nullable = true)
 |    |-- ab_group: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- revenue: double (nullable = true)
 |-- category: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- signup_date: string (nullable = true)
 |-- plan: string (nullable = true)
 |-- marketing_opt_in: boolean (nullable = true)

+--

## Task 8 — Group by date, country, category and compute KPIs

In [9]:
daily_kpi = enriched_df.groupBy("date", F.col("context.country").alias("country"), "category").agg(
    F.count("*").alias("events_total"),
    F.count(F.when(F.col("event") == "purchase", True)).alias("purchases"),
    F.sum("revenue").alias("revenue"),
    F.countDistinct("user_id").alias("unique_users"),
)

daily_kpi.orderBy("date", "country", "category").show(20)

+----------+-------+-----------+------------+---------+------------------+------------+
|      date|country|   category|events_total|purchases|           revenue|unique_users|
+----------+-------+-----------+------------+---------+------------------+------------+
|2026-02-07|     BR|      books|          44|        1|            234.78|          44|
|2026-02-07|     BR|electronics|          40|        2|            348.81|          40|
|2026-02-07|     BR|    fashion|          31|        1|            168.69|          31|
|2026-02-07|     BR|       home|          43|        1|              9.28|          43|
|2026-02-07|     BR|     sports|          44|        1|             46.92|          44|
|2026-02-07|     BR|       toys|          40|        0|               0.0|          40|
|2026-02-07|     DE|      books|          51|        2|254.64999999999998|          51|
|2026-02-07|     DE|electronics|          43|        2|             58.75|          43|
|2026-02-07|     DE|    fashion|

## Task 9 — 7-day rolling revenue using window functions

In [10]:
# Window: partition by country & category, order by date, frame = current row + 6 preceding rows
w = Window.partitionBy("country", "category") \
          .orderBy("date") \
          .rowsBetween(-6, 0)

daily_kpi = daily_kpi.withColumn("revenue_7d", F.sum("revenue").over(w))

daily_kpi.orderBy("date", "country", "category").show(20)

+----------+-------+-----------+------------+---------+------------------+------------+------------------+
|      date|country|   category|events_total|purchases|           revenue|unique_users|        revenue_7d|
+----------+-------+-----------+------------+---------+------------------+------------+------------------+
|2026-02-07|     BR|      books|          44|        1|            234.78|          44|            234.78|
|2026-02-07|     BR|electronics|          40|        2|            348.81|          40|            348.81|
|2026-02-07|     BR|    fashion|          31|        1|            168.69|          31|            168.69|
|2026-02-07|     BR|       home|          43|        1|              9.28|          43|              9.28|
|2026-02-07|     BR|     sports|          44|        1|             46.92|          44|             46.92|
|2026-02-07|     BR|       toys|          40|        0|               0.0|          40|               0.0|
|2026-02-07|     DE|      books|     

## Task 10 — Write to Parquet partitioned by date

In [11]:
OUTPUT_DIR = "../out/daily_kpi"

daily_kpi.write \
    .partitionBy("date") \
    .mode("overwrite") \
    .parquet(OUTPUT_DIR)

print(f"Written to {OUTPUT_DIR}")

Written to ../out/daily_kpi


## Task 11 — Inspect results: list partitions and read back one

In [12]:
import os

# List date-partition folders
partitions = sorted([d for d in os.listdir(OUTPUT_DIR) if d.startswith("date=")])
print(f"Total date partitions: {len(partitions)}")
print("First 10:", partitions[:10])

# Read back one partition and display
first_partition = partitions[0]
one_day = spark.read.parquet(os.path.join(OUTPUT_DIR, first_partition))
print(f"\nSample from {first_partition}:")
one_day.show(truncate=False)

Total date partitions: 71
First 10: ['date=2026-02-07', 'date=2026-02-08', 'date=2026-02-09', 'date=2026-02-10', 'date=2026-02-11', 'date=2026-02-12', 'date=2026-02-13', 'date=2026-02-14', 'date=2026-02-15', 'date=2026-02-16']

Sample from date=2026-02-07:
+-------+-----------+------------+---------+------------------+------------+------------------+
|country|category   |events_total|purchases|revenue           |unique_users|revenue_7d        |
+-------+-----------+------------+---------+------------------+------------+------------------+
|BR     |books      |44          |1        |234.78            |44          |234.78            |
|BR     |electronics|40          |2        |348.81            |40          |348.81            |
|BR     |fashion    |31          |1        |168.69            |31          |168.69            |
|BR     |home       |43          |1        |9.28              |43          |9.28              |
|BR     |sports     |44          |1        |46.92             |44      

## Task 12 — Repartition drill

We experiment with repartitioning at three stages and observe the effect on task counts and output files.

### 12a — Repartition before joining (by user_id)

In [13]:
import time

num_cores = os.cpu_count()
print(f"Available CPU cores: {num_cores}")

# Repartition events by user_id before joins
events_repart_user = events_df.repartition(num_cores, "user_id")
print(f"Events partitions after repartition by user_id: {events_repart_user.rdd.getNumPartitions()}")

t0 = time.time()
enriched_a = events_repart_user \
    .join(F.broadcast(items_df), on="item_id", how="left") \
    .join(F.broadcast(users_df), events_repart_user["user_id"] == users_df["id"], how="left") \
    .drop(users_df["country"]).drop(users_df["id"])
count_a = enriched_a.count()
t1 = time.time()
print(f"12a - Join after repartition by user_id: {count_a} rows, {t1-t0:.2f}s")

Available CPU cores: 8
Events partitions after repartition by user_id: 8
12a - Join after repartition by user_id: 199995 rows, 0.38s


### 12b — Repartition before aggregation (by date, country, category)

In [14]:
# Repartition enriched data by the groupBy keys before aggregation
enriched_repart = enriched_df.repartition(num_cores, "date", F.col("context.country"), "category")
print(f"Enriched partitions after repartition by (date, country, category): {enriched_repart.rdd.getNumPartitions()}")

t0 = time.time()
daily_kpi_b = enriched_repart.groupBy("date", F.col("context.country").alias("country"), "category").agg(
    F.count("*").alias("events_total"),
    F.count(F.when(F.col("event") == "purchase", True)).alias("purchases"),
    F.sum("revenue").alias("revenue"),
    F.countDistinct("user_id").alias("unique_users"),
)
count_b = daily_kpi_b.count()
t1 = time.time()
print(f"12b - Aggregation after repartition by groupBy keys: {count_b} rows, {t1-t0:.2f}s")

Enriched partitions after repartition by (date, country, category): 8
12b - Aggregation after repartition by groupBy keys: 3338 rows, 0.58s


### 12c — Repartition / coalesce before write

In [15]:
OUTPUT_DIR_REPART = "../out/daily_kpi_coalesced"

# coalesce(1) merges all partitions within each date partition into a single file
# This reduces the number of output files (good for small result sets) but
# funnels all data through one task per partition.
t0 = time.time()
daily_kpi.coalesce(1) \
    .write \
    .partitionBy("date") \
    .mode("overwrite") \
    .parquet(OUTPUT_DIR_REPART)
t1 = time.time()
print(f"12c - Write with coalesce(1): {t1-t0:.2f}s")

# Count output files
import glob as g
default_files = g.glob(f"{OUTPUT_DIR}/**/*.parquet", recursive=True)
coalesced_files = g.glob(f"{OUTPUT_DIR_REPART}/**/*.parquet", recursive=True)
print(f"Default write: {len(default_files)} parquet files")
print(f"Coalesced write: {len(coalesced_files)} parquet files")

12c - Write with coalesce(1): 1.49s
Default write: 71 parquet files
Coalesced write: 71 parquet files


In [16]:
spark.stop()